# Proof of Stake（Ethereum）

EthereumのPoSは **Gasper** と呼ばれ、次の2つを組み合わせている。

- **LMD GHOST**：分岐したチェーンのうち、どれを先端（head）とするかを決めるフォークチョイスルール
- **Casper FFG**：チェックポイントを確定（finalize）させるファイナリティの仕組み

PoS一般の考え方は [コンセンサスアルゴリズム](../blockchain/consensus.ipynb) を参照。

## The Merge

Ethereumは2022年9月の **The Merge** でPoWからPoSに移行した。

2020年12月からPoSのチェーン（**Beacon Chain**）をメインネットと並行して稼働させておき、The Mergeの時点で既存のPoWチェーンをBeacon Chainに合流させた。飛行中の飛行機のエンジンを交換するような移行と例えられる。これにより、Ethereumの電力消費は99.9%以上減ったとされる。

## 2層構造

The Merge以降、Ethereumのノードは2つのソフトウェアで構成される。

```mermaid
flowchart TB
    subgraph node["Ethereumのノード"]
        CL["コンセンサスクライアント<br/>（Consensus Layer）<br/>バリデータ管理・投票・フォークチョイス"]
        EL["実行クライアント<br/>（Execution Layer）<br/>トランザクション実行・ステート管理"]
        CL <-->|"Engine API"| EL
    end
    V["バリデータクライアント<br/>（署名鍵を保持）"] --> CL
```

| レイヤー | 役割 | 管理するステート |
| --- | --- | --- |
| Consensus Layer | 正しい状態遷移の記録について合意する | ビーコンステート（バリデータの一覧、残高、スロットなど） |
| Execution Layer | トランザクションを実行・検証する | ワールドステート（アカウントの残高、コントラクトの変数など） |

両者は相互に依存している。コンセンサスクライアントだけではトランザクションを実行・検証できず、実行クライアントだけではどのチェーンが正統か分からない。

## スロットとエポック

時間は次の単位で区切られる。

- **スロット（slot）**：12秒。各スロットで1人のバリデータがブロックを提案する（提案されずに空になることもある）
- **エポック（epoch）**：32スロット = 6.4分。投票の割り当てやファイナリティ、報酬の計算はエポック単位で行う

## バリデータ

### バリデータになる

ETHをデポジットコントラクト（実行レイヤー上のコントラクト）に預けると、**バリデータ** として登録される。バリデータの役割は2つ。

- **ブロック提案（proposal）**：割り当てられたスロットで新しいブロックを作る
- **投票（attestation）**：チェーンの先端とチェックポイントについて投票する

1つのバリデータを有効化するには32 ETHが必要。もともと1バリデータの有効残高の上限も32 ETHで、64 ETHを預けたい場合は2つのバリデータを立てる必要があった。2025年のPectraアップグレード（EIP-7251）以降は、1バリデータで最大2,048 ETHまで有効残高を持てるようになった（compounding validator）。

ステーク総額は3,000万ETH超（総供給量の約4分の1）に達している（2025年時点）。

### 鍵

バリデータはEOAとは別の鍵を使う。

| 鍵 | 用途 | 形式 |
| --- | --- | --- |
| 署名鍵（signing key） | ブロック提案と投票の署名 | BLS署名 |
| 引き出し先（withdrawal credentials） | ステークと報酬の引き出し先 | 通常はEOAのアドレスを指定 |

**BLS署名** は、多数の署名を1つに **集約（aggregate）** できるのが特徴。数十万のバリデータの投票を、ブロックにそのまま全部載せずに済む。

### ライフサイクル

```mermaid
flowchart LR
    D["デポジット<br/>（実行レイヤー）"] --> P["有効化待ち<br/>（pending）"] --> A["アクティブ<br/>提案・投票"] --> E["退出<br/>（exit）"] --> W["引き出し可能<br/>（withdrawable）"]
    A -->|"スラッシング"| E
```

- 新規の有効化や退出は、1エポックあたりの人数（チャーン上限）が制限されており、大量の出入りが一度に起きないようになっている
- アクティブになってから約27時間（256エポック）経過すれば、自発的に退出できる
- 2023年のShapellaアップグレードまで、ステークしたETHは引き出せなかった

## 提案者と投票者の選出

### 選出方法

- **提案者**：各スロットに1人。有効残高に比例した確率で選ばれる
- **投票者**：各エポックで、全バリデータがいずれか1つのスロットの **委員会（committee）** に割り当てられる。1委員会あたり最低128人を目標に、1スロットあたり最大64委員会

つまり、すべてのアクティブなバリデータは各エポックでちょうど1回投票する。委員会の上限は2,048人なので、1エポックに割り当てられる投票者の上限は $2048 \times 64 \times 32 \approx 419$ 万人になる。

### RANDAO

選出にはランダム性が必要だが、分散システムで偏りのない乱数を作るのは難しい。外部の乱数源を使うとその提供者を信用しなければならない。

Ethereumは **RANDAO** という方式で乱数を作る。各ブロックの提案者が、エポック番号に対する自分のBLS署名（`randao_reveal`）を提出し、それを累積的にXORで混ぜ合わせていく。

- 署名は決定論的なので、提案者が値を自由に選ぶことはできない
- 各提案者は「ブロックを提案しない」ことで乱数に1ビット程度の影響を与えられるが、その代償として報酬を失う

この乱数は、2エポック程度先の提案者・委員会の割り当てや、EVMの `PREVRANDAO` オペコードとして使われる。

## ブロックの作成

提案者は、実行レイヤーと協調してブロックを作る。

1. フォークチョイスルールで現在の正統なチェーンの先端を決める
2. **実行クライアント** がmempoolからトランザクションを選んで実行し、結果を **Execution Payload** として返す
3. **コンセンサスクライアント** が、attestationプールから集約済みの投票を選び、バリデータ関連の処理を行う
4. ブロック全体に署名して配信する

### ブロックの構造

```mermaid
flowchart TB
    subgraph SBB["SignedBeaconBlock"]
        subgraph BB["BeaconBlock"]
            H["slot, proposer_index,<br/>parent_root, state_root"]
            subgraph Body["BeaconBlockBody"]
                C["randao_reveal, graffiti,<br/>attestations, deposits,<br/>voluntary_exits, slashings, ..."]
                subgraph EP["ExecutionPayload"]
                    X["parent_hash, fee_recipient,<br/>state_root, receipts_root,<br/>base_fee_per_gas, block_hash,<br/>transactions, withdrawals"]
                end
            end
        end
        S["signature（提案者のBLS署名）"]
    end
```

PoW時代のブロックに相当する情報は、Execution Payloadとしてビーコンブロックの中に包含されている。コンセンサスレイヤーのデータ構造のハッシュには、MPTではなく **SSZ**（Simple Serialize）のマークル化が使われる。

### 1スロットの流れ

| タイミング | 処理 |
| --- | --- |
| 0秒 | 提案者がブロックを配信 |
| 4秒 | 委員会のバリデータが、その時点で見えているチェーンの先端に投票する（ブロックが届いていなければ前のブロックに投票する） |
| 8秒 | 集約担当（aggregator）が委員会内の投票を集約して配信 |
| 次のスロット以降 | 集約された投票が後続のブロックに取り込まれる |

## 投票（attestation）

投票には2種類の情報が含まれる。

| 項目 | 内容 | 用途 |
| --- | --- | --- |
| `beacon_block_root` | 自分が先端だと考えるブロック | LMD GHOST（フォークチョイス） |
| `source` | 直近でjustifiedされたチェックポイント | Casper FFG（ファイナリティ） |
| `target` | 現在のエポックのチェックポイント | Casper FFG（ファイナリティ） |

「投票」というと人気投票のような印象を受けるが、実際には「自分が検証した結果、これが正しいチェーンである」という証言（attestation）である。

## LMD GHOST：フォークチョイスルール

**GHOST（Greedy Heaviest Observed SubTree）** は、根から出発して、各分岐点で「その先の部分木が最も多くの票を集めている子」へ進むことを繰り返し、たどり着いた葉を先端とするルール。**LMD（Latest Message Driven）** は、各バリデータの **最新の** 投票だけを数えるという意味。

- 各票の重みはそのバリデータの有効残高
- 探索は直近のjustifiedチェックポイントから開始し、finalizedより前で分岐した枝は候補にしない

Bitcoinの最長チェーンルールが「ブロックの数（作業量）」で比べるのに対し、LMD GHOSTは「その枝を支持するステークの量」で比べる。

In [1]:
from collections import defaultdict


def lmd_ghost(parent: dict[str, str], latest_votes: dict[str, str], weights: dict[str, float], root: str) -> str:
    """各バリデータの最新の投票先と重みから、先端ブロックを求める"""
    children = defaultdict(list)
    for block, p in parent.items():
        children[p].append(block)

    def subtree_weight(block: str) -> float:
        # block を祖先に持つブロックへの票の合計
        total = 0.0
        for v, target in latest_votes.items():
            b = target
            while b is not None:
                if b == block:
                    total += weights[v]
                    break
                b = parent.get(b)
        return total

    head = root
    while children[head]:
        head = max(children[head], key=lambda c: (subtree_weight(c), c))
    return head


# A - B - C - D
#      \
#       C' - D' - E'
parent = {"B": "A", "C": "B", "D": "C", "C'": "B", "D'": "C'", "E'": "D'"}
weights = {f"v{i}": 32.0 for i in range(5)}

votes = {"v0": "D", "v1": "D", "v2": "C", "v3": "E'", "v4": "D'"}
print("先端:", lmd_ghost(parent, votes, weights, "A"), "（短い枝でも票が多ければ選ばれる）")

votes["v2"] = "E'"  # v2 が新しい投票で乗り換え（古い票は捨てられる）
print("先端:", lmd_ghost(parent, votes, weights, "A"))

先端: D （短い枝でも票が多ければ選ばれる）
先端: E'


## Casper FFG：ファイナリティ

### チェックポイント

各エポックの最初のスロットのブロックを **チェックポイント** と呼ぶ（そのスロットが空なら直前のブロック）。投票の `source → target` は、チェックポイント間のリンクへの投票になっている。

### justifiedとfinalized

1. あるリンク `source → target` に、全ステークの **2/3以上** の票が集まると、`target` が **justified** になる
2. 連続するエポックで、justifiedなチェックポイントの次のチェックポイントもjustifiedになると、前者が **finalized** になる

```mermaid
flowchart LR
    E1["エポック N<br/>justified → finalized"] -->|"2/3以上の投票"| E2["エポック N+1<br/>justified"] --> E3["エポック N+2<br/>投票中"]
```

finalizedされたブロックは覆らない。覆すには全ステークの1/3以上が矛盾する投票をする必要があり、その場合それらのバリデータはスラッシングされる。つまり、**ファイナリティを覆すコストは少なくともステーク総額の1/3** という形で、安全性を経済的に定量化できる。これを **経済的ファイナリティ（economic finality）** と呼ぶ。

通常、ブロックは約2〜3エポック（約13〜19分）でfinalizedされる。

### 2つの仕組みを組み合わせる理由

- LMD GHOSTは、ネットワークが不安定でもチェーンを伸ばし続けられる（活性を優先）
- Casper FFGは、確定したブロックが覆らないことを保証する（安全性を優先）

数十万〜百万のバリデータ全員で毎ブロックPBFT型の合意をすると遅すぎる。そこで、委員会単位の投票で12秒ごとにブロックを進めつつ（可用性）、時間のかかるネットワーク全体での確定を並行してゆっくり進める（安全性）、という設計になっている。

:::{note}
ファイナリティまで十数分かかるのはEthereumの課題の1つで、1スロットで確定させる **Single Slot Finality（SSF）** や、それに向けた3スロットファイナリティなどが研究されている。
:::

## 報酬とペナルティ

| 区分 | 内容 |
| --- | --- |
| 投票の報酬 | 正しいsource・target・headに、遅れずに投票すると得られる。報酬の大部分を占める |
| 提案の報酬 | ブロックに投票を含めた分の報酬 + トランザクションのpriority fee + MEV（→ MEV-Boost） |
| 同期委員会の報酬 | ライトクライアント向けの署名を行う委員会（512人、約27時間ごとに交代）に選ばれた場合 |
| 投票しなかった場合 | 得られたはずの報酬と同程度のペナルティ。オフラインになっても損失は小さい |
| スラッシング | 同じスロットで2つの異なるブロックを提案、同じtargetエポックに2つの異なる投票（double vote）、他の投票を囲む投票（surround vote）をした場合。ステークの一部を没収され、強制退出させられる。同時期にスラッシングされたバリデータが多いほど没収額が増える（correlation penalty） |
| Inactivity leak | 4エポック以上ファイナリティが得られない場合、投票していないバリデータのステークを徐々に減らし、オンラインのバリデータが2/3を回復できるようにする |

correlation penaltyにより、単独のミスによるスラッシングは小さく、多数のバリデータが結託した攻撃は大きく罰せられる。Inactivity leakは、大規模なネットワーク分断が起きても最終的にファイナリティを回復するための仕組み。

## ステーキングの形態

32 ETHを用意して自分でノードを24時間運用するのはハードルが高いため、様々な形態がある。

| 形態 | 概要 | 例 |
| --- | --- | --- |
| ソロステーキング | 自分でノードを運用し、鍵も自分で管理する | — |
| ステーキングサービス | ノード運用を業者に委託する | — |
| リキッドステーキング | ETHを預けると、ステーク中のETHを表すトークン（LST）を受け取れる。LSTはDeFiで運用できる | Lido（stETH）, Rocket Pool（rETH） |
| 取引所 | 取引所にETHを預け、取引所がステークする | Coinbase, Binance |
| リステーキング | ステーク済みのETHを、他のプロトコルのセキュリティ担保として再利用する | EigenLayer |

リキッドステーキングは利便性が高い一方、特定のプロトコルにステークが集中することによる中央集権化のリスクが指摘されている。

## 参考文献

- [Proof-of-stake (PoS) | ethereum.org](https://ethereum.org/developers/docs/consensus-mechanisms/pos/)
- Ben Edgington. [Upgrading Ethereum](https://eth2book.info/)
- Buterin, V. et al. (2020). [Combining GHOST and Casper](https://arxiv.org/abs/2003.03052)
- Buterin, V. & Griffith, V. (2017). [Casper the Friendly Finality Gadget](https://arxiv.org/abs/1710.09437)
- [beaconcha.in](https://beaconcha.in/)（バリデータやエポックの状況を確認できるエクスプローラー）